
# Week 11: Vector Embeddings and Semantic Search

## 1. Project Overview

This project demonstrates how vector embeddings are used to find relevant information from text documents.

We use Artificial Intelligence and Data Science as the document domain. Sentence Transformers generate vector embeddings, and ChromaDB stores the embeddings for semantic search.

We also implement keyword-based search and compare both search methods.

## 2. Project Objectives

- Load and process text documents.
- Generate vector embeddings using Sentence Transformers.
- Store embeddings in ChromaDB.
- Perform semantic search using user queries.
- Implement keyword-based search.
- Compare semantic search and keyword search results.

## 3. Technologies Used

- Python
- Sentence Transformers
- ChromaDB
- Pandas
- NumPy
- Jupyter Notebook
- VS Code



from pathlib import Path
import pandas as pd

# Set project paths
BASE_DIR = Path.cwd()

if not (BASE_DIR / "dataset").exists():
    BASE_DIR = BASE_DIR.parent

DOC_DIR = BASE_DIR / "dataset" / "documents"
QUERY_FILE = BASE_DIR / "dataset" / "queries.csv"

# Load text documents
document_files = sorted(DOC_DIR.glob("*.txt"))

documents = []

for file_path in document_files:
    content = file_path.read_text(encoding="utf-8-sig").strip()

    lines = [line.strip() for line in content.splitlines() if line.strip()]

    title_line = next(
        (line for line in lines if line.lower().startswith("title:")),
        None
    )

    title = title_line.split(":", 1)[1].strip() if title_line else file_path.stem

    documents.append({
        "document_id": file_path.stem,
        "title": title,
        "content": content
    })

# Load search queries
queries_df = pd.read_csv(QUERY_FILE)

# Display dataset information
documents_df = pd.DataFrame(documents)

print("Total documents loaded:", len(documents_df))
print("Total queries loaded:", len(queries_df))

display(documents_df[["document_id", "title"]])
display(queries_df)



import numpy as np
from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract document contents
document_texts = documents_df["content"].tolist()

# Generate vector embeddings
embeddings = model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Save embeddings
EMBEDDINGS_FILE = BASE_DIR / "dataset" / "embeddings.npy"
np.save(EMBEDDINGS_FILE, embeddings)

print("Vector embeddings generated successfully!")
print("Number of documents:", len(embeddings))
print("Embedding shape:", embeddings.shape)
print("Saved at:", EMBEDDINGS_FILE)


In [ ]:

import numpy as np
from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract document contents
document_texts = documents_df["content"].tolist()

# Generate vector embeddings
embeddings = model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Save embeddings
EMBEDDINGS_FILE = BASE_DIR / "dataset" / "embeddings.npy"
np.save(EMBEDDINGS_FILE, embeddings)

print("Vector embeddings generated successfully!")
print("Number of documents:", len(embeddings))
print("Embedding shape:", embeddings.shape)
print("Saved at:", EMBEDDINGS_FILE)


In [ ]:

import numpy as np
from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract document contents
document_texts = documents_df["content"].tolist()

# Generate vector embeddings
embeddings = model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Save embeddings
EMBEDDINGS_FILE = BASE_DIR / "dataset" / "embeddings.npy"
np.save(EMBEDDINGS_FILE, embeddings)

print("Vector embeddings generated successfully!")
print("Number of documents:", len(embeddings))
print("Embedding shape:", embeddings.shape)
print("Saved at:", EMBEDDINGS_FILE)


In [ ]:

import chromadb

# Create persistent ChromaDB storage
DB_DIR = BASE_DIR / "chroma_db"

client = chromadb.PersistentClient(path=str(DB_DIR))

# Create or access collection
collection = client.get_or_create_collection(
    name="ai_data_science_documents",
    metadata={"hnsw:space": "cosine"}
)

# Store documents and their embeddings
metadatas = [
    {
        "title": row["title"],
        "file_name": f"{row['document_id']}.txt"
    }
    for _, row in documents_df.iterrows()
]

collection.upsert(
    ids=documents_df["document_id"].tolist(),
    documents=document_texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents stored successfully in ChromaDB!")
print("Total documents:", collection.count())
